## 1º Carregar o Dataset
 **Importação dos Dados:** 
  * Realizar a leitura do arquivo `.csv` que está no projeto na pasta `1_DataSet` o arquivo está como `.zip` utilizando bibliotecas adequadas (ex: `pd.read_csv()` no Pandas).
**Inspeção Inicial:**
  * Verificação das primeiras e últimas linhas (`head()` / `tail()`).
  * Análise preliminar de dimensões (linhas e colunas) e tipos de dados estruturais (`info()`).

## Vamos ler diretamente para o Pandas (Sem descompactar no disco)
Esta é uma boa prática para evitar redundância de arquivos e economizar espaço de armazenamento:

In [ ]:
import pandas as pd
import io
import zipfile

In [97]:
DataSet_INEP_zip = 'C:/Users/Bruno/Desktop/Pos_SENAC_DF/Aprendizagem_de_maquina_supervisionado/Trabalho_Final/1_DataSet/Tabela_Escola_2025_INEP.zip'
DataSet_IDEB_zip = 'C:/Users/Bruno/Desktop/Pos_SENAC_DF/Aprendizagem_de_maquina_supervisionado/Trabalho_Final/1_DataSet/IDEB_Escolas_Ensino_medio_2025.zip'

In [98]:
def carregar_dados(DataSet):
    dados = None
    try:
        with zipfile.ZipFile(DataSet, 'r') as zip_ref:
            nome_csv = [f for f in zip_ref.namelist() if f.endswith('.csv')][0]
            with zip_ref.open(nome_csv) as arquivo_csv:
                dados = pd.read_csv(arquivo_csv, sep=None, engine='python', encoding='latin1')
    except:
        print("Erro ao carregar os dados do arquivo ZIP.")
    return dados

## Vamos inicialmente trabalhar com o DataSet com os dados do INEP, esse DataSet contém dados pertinentes á todas as escolas que participaram do Censo Escolar de 2025.
**Vamos inicialmente carregar o DataSet e realizar uma limpeza de algumas features que não serão consideradas nas etapas seguinte.**
*É Importante nessa etapa o acompanhamento das descrições que compõem cada features, essa descrição está em 01.Entendimento_dos_dados.ipynb


In [68]:
df_INEP = carregar_dados(DataSet_INEP_zip)
print("DataSet_INEP_zip carregado com sucesso!")
print("\n DataSet_INEP carregado com sucesso!\n\n")
df_INEP.info()
print("\n\nResumo estatístico do DataFrame df_INEP:\n")
df_INEP.describe(include='all')


DataSet_INEP_zip carregado com sucesso!

 DataSet_INEP carregado com sucesso!


<class 'pandas.DataFrame'>
RangeIndex: 214192 entries, 0 to 214191
Columns: 290 entries, NU_ANO_CENSO to IN_ESP_EXCLUSIVA_PROF
dtypes: float64(268), int64(10), str(12)
memory usage: 473.9 MB


Resumo estatístico do DataFrame df_INEP:



,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,...,IN_ESP_EXCLUSIVA_MEDIO_FIC,IN_ESP_EXCLUSIVA_MEDIO_NORMAL,IN_COMUM_EJA_FUND,IN_COMUM_EJA_MEDIO,IN_COMUM_EJA_PROF,IN_ESP_EXCLUSIVA_EJA_FUND,IN_ESP_EXCLUSIVA_EJA_MEDIO,IN_ESP_EXCLUSIVA_EJA_PROF,IN_COMUM_PROF,IN_ESP_EXCLUSIVA_PROF
count,214192.0,214191,214192.000000,214191,214191,214192.000000,214191,2.141920e+05,214191,214191.000000,...,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000,180540.000000
unique,NaN,5,NaN,27,27,NaN,5298,NaN,133,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,Sudeste,NaN,São Paulo,SP,NaN,São Paulo,NaN,São Paulo,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,74567,NaN,33616,33616,NaN,7905,NaN,15338,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2025.0,NaN,2.660459,NaN,NaN,30.348944,NaN,3.050211e+06,NaN,3038.258741,...,0.000039,0.000011,0.129052,0.048444,0.010009,0.006735,0.000343,0.000061,0.033184,0.000255
std,0.0,NaN,1.025717,NaN,NaN,9.480302,NaN,9.517342e+05,NaN,948.387129,...,0.006227,0.003328,0.335258,0.214702,0.099543,0.081793,0.018528,0.007805,0.179117,0.015960
min,2025.0,NaN,1.000000,NaN,NaN,11.000000,NaN,1.100015e+06,NaN,1101.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025.0,NaN,2.000000,NaN,NaN,23.000000,NaN,2.313757e+06,NaN,2306.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2025.0,NaN,3.000000,NaN,NaN,31.000000,NaN,3.119856e+06,NaN,3102.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2025.0,NaN,3.000000,NaN,NaN,35.000000,NaN,3.548500e+06,NaN,3504.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Vamos retirar do DataSet as features que não são necessárias e selecionar somente os dados das escolas públicas, pois a nota do Ideb é uma avaliação somente de escolas públicas.

In [ ]:
print("\n\nRemovendo variáveis:\n")

variaveis_para_remover = [
    "NU_ANO_CENSO",
    "NO_REGIAO",
    "CO_REGIAO",
    "NO_UF",
    "NO_MUNICIPIO",
    "CO_MUNICIPIO",
    "NO_REGIAO_GEOG_INTERM",
    "CO_REGIAO_GEOG_INTERM",
    "NO_REGIAO_GEOG_IMED",
    "CO_REGIAO_GEOG_IMED",
    "NO_MESORREGIAO",
    "CO_MESORREGIAO",
    "NO_MICRORREGIAO",
    "CO_MICRORREGIAO",
    "NO_DISTRITO",
    "CO_DISTRITO",
    "NO_SUBDISTRITO",
    "CO_SUBDISTRITO",
    "NO_REGIAO_ADMINISTRATIVA",
    "CO_REGIAO_ADMINISTRATIVA",
    "TP_CATEGORIA_ESCOLA_PRIVADA",
    "TP_LOCALIZACAO_DIFERENCIADA",
    "DS_ENDERECO",
    "NU_ENDERECO",
    "DS_COMPLEMENTO",
    "NO_BAIRRO",
    "CO_CEP",
    "NU_DDD",
    "NU_TELEFONE",
    "LATITUDE",
    "LONGITUDE",
    "TP_SITUACAO_FUNCIONAMENTO",
    "CO_ORGAO_REGIONAL",
    "DT_ANO_LETIVO_INICIO",
    "DT_ANO_LETIVO_TERMINO",
    "IN_VINCULO_SECRETARIA_EDUCACAO",
    "IN_VINCULO_SEGURANCA_PUBLICA",
    "IN_VINCULO_SECRETARIA_SAUDE",
    "IN_VINCULO_OUTRO_ORGAO",
    "IN_PODER_PUBLICO_PARCERIA",
    "TP_PODER_PUBLICO_PARCERIA",
    "IN_CONVENIADA_PP",
    "TP_CONVENIO_PODER_PUBLICO",
    "IN_FORMA_CONT_TERMO_COLABORA",
    "IN_FORMA_CONT_TERMO_FOMENTO",
    "IN_FORMA_CONT_ACORDO_COOP",
    "IN_FORMA_CONT_PRESTACAO_SERV",
    "IN_FORMA_CONT_COOP_TEC_FIN",
    "IN_FORMA_CONT_CONSORCIO_PUB",
    "IN_FORMA_CONT_MU_TERMO_COLAB",
    "IN_FORMA_CONT_MU_TERMO_FOMENTO",
    "IN_FORMA_CONT_MU_ACORDO_COOP",
    "IN_FORMA_CONT_MU_PREST_SERV",
    "IN_FORMA_CONT_MU_COOP_TEC_FIN",
    "IN_FORMA_CONT_MU_CONSORCIO_PUB",
    "IN_FORMA_CONT_ES_TERMO_COLAB",
    "IN_FORMA_CONT_ES_TERMO_FOMENTO",
    "IN_FORMA_CONT_ES_ACORDO_COOP",
    "IN_FORMA_CONT_ES_PREST_SERV",
    "IN_FORMA_CONT_ES_COOP_TEC_FIN",
    "IN_FORMA_CONT_ES_CONSORCIO_PUB",
    "IN_TIPO_ATEND_ESCOLARIZACAO",
    "IN_TIPO_ATEND_AC",
    "IN_TIPO_ATEND_AEE",
    "IN_MANT_ESCOLA_PRIVADA_EMP",
    "IN_MANT_ESCOLA_PRIVADA_ONG",
    "IN_MANT_ESCOLA_PRIVADA_OSCIP",
    "IN_MANT_ESCOLA_PRIV_ONG_OSCIP",
    "IN_MANT_ESCOLA_PRIVADA_SIND",
    "IN_MANT_ESCOLA_PRIVADA_SIST_S",
    "IN_MANT_ESCOLA_PRIVADA_S_FINS",
    "NU_CNPJ_ESCOLA_PRIVADA",
    "NU_CNPJ_MANTENEDORA",
    "TP_REGULAMENTACAO",
    "TP_RESPONSAVEL_REGULAMENTACAO",
    "CO_ESCOLA_SEDE_VINCULADA",
    "CO_IES_OFERTANTE",
    "IN_LOCAL_FUNC_OUTROS"
]

df_INEP_limpo = df_INEP.drop(columns=variaveis_para_remover, errors='ignore')
df_INEP_limpo_rede_publica = df_INEP_limpo[df_INEP_limpo['CO_REDE'] == 1]
df_INEP_limpo_rede_publica.drop_duplicates(inplace=True)

print("\n\nDataFrame df_INEP_limpo_rede_publica criado com sucesso!\n")
df_INEP_limpo_rede_publica.info()


## Trabalhando com dados do resultado do Ideb.

In [101]:
df_IDEB= carregar_dados(DataSet_IDEB_zip)
print("DataSet_IDEB_zip carregado com sucesso!")
print("\n DataSet_IDEB carregado com sucesso!\n\n")
df_IDEB.info()
print("\n\nResumo estatístico do DataFrame df_IDEB:\n")
df_IDEB.describe(include='all')

DataSet_IDEB_zip carregado com sucesso!

 DataSet_IDEB carregado com sucesso!


<class 'pandas.DataFrame'>
RangeIndex: 22171 entries, 0 to 22170
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Sigla da UF          22171 non-null  str  
 1   Código do Município  22171 non-null  int64
 2   Nome do Município    22171 non-null  str  
 3   Código da Escola     22171 non-null  int64
 4   Nome da Escola       22171 non-null  str  
 5   Rede                 22171 non-null  str  
 6   IDEB_2025            22170 non-null  str  
dtypes: int64(2), str(5)
memory usage: 1.2 MB


Resumo estatístico do DataFrame df_IDEB:



,Sigla da UF,Código do Município,Nome do Município,Código da Escola,Nome da Escola,Rede,IDEB_2025
count,22171,2.217100e+04,22171,2.217100e+04,22171,22171,22170
unique,27,NaN,5291,NaN,21403,4,65
top,SP,NaN,São Paulo,NaN,EE DE ENSINO MEDIO,Estadual,-
freq,4322,NaN,710,NaN,18,20081,4587
mean,NaN,3.236898e+06,NaN,3.234697e+07,NaN,NaN,NaN
std,NaN,9.821404e+05,NaN,9.793113e+06,NaN,NaN,NaN
min,NaN,1.100015e+06,NaN,1.100006e+07,NaN,NaN,NaN
25%,NaN,2.604858e+06,NaN,2.605412e+07,NaN,NaN,NaN
50%,NaN,3.300407e+06,NaN,3.300831e+07,NaN,NaN,NaN
75%,NaN,4.100400e+06,NaN,4.100148e+07,NaN,NaN,NaN


In [ ]:
print("\n\nRemovendo dados de escolas privadas e de escolas com IDEB não disponível:\n")

df_filtrado = df_IDEB[(df_IDEB['IDEB_2025'].notna()) & (df_IDEB['IDEB_2025'].str.strip() != '-')]
df_filtrado.drop_duplicates(inplace=True)
df_filtrado.info()